# **1. Install Required Libraries**

In [ ]:
!pip -q install -U openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.6 MB/s eta 0:00:00


# **2. Configure OpenAI API**

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GENERATION_MODEL = "gpt-4.1-mini"

print("OpenAI client configured successfully.")
print("Model:", GENERATION_MODEL)


Enter your OpenAI API key: ··········
OpenAI client configured successfully.
Model: gpt-4.1-mini


# **3. First Tool — A Python Function**

A tool starts with a function that performs a specific task.


In [ ]:
def calculate_leave_balance(total_leaves, leaves_taken):
    return {"remaining_leaves": total_leaves - leaves_taken}

result = calculate_leave_balance(24, 7)
print(result)

{'remaining_leaves': 17}


# **4. Function Calling - IMPORTANT NOTES**

1. The function exists in our application, but the **LLM does not execute Python code**.

2. The model receives a description of the available function and can return a **tool call** with the required arguments.

3. The application then executes the function and sends the result back to the model.


## **4.1 Define the Tool**

In [ ]:
tools = [
    {
        "type": "function",
        "name": "calculate_leave_balance",
        "description": "Calculate an employee's remaining leave balance.",
        "parameters": {
            "type": "object",
            "properties": {
                "total_leaves": {
                    "type": "integer",
                    "description": "Total annual leave allowance."
                },
                "leaves_taken": {
                    "type": "integer",
                    "description": "Number of leaves already taken."
                }
            },
            "required": ["total_leaves", "leaves_taken"],
            "additionalProperties": False
        }
    }
]

## **4.2 Send the User Request to the Model**

In [ ]:
question = "An employee has 24 total leaves and has taken 8. What is the remaining leave balance?"

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question,
                                    tools=tools)
response.output

[ResponseFunctionToolCall(arguments='{"total_leaves":24,"leaves_taken":8}', call_id='call_aGLCZ7I3KBE0bzRr0X9TqCTq', name='calculate_leave_balance', type='function_call', id='fc_089495b29be33e2d006aaccfd611cc87d095459a896450b674', async_=None, caller=None, namespace=None, status='completed')]

In [ ]:
print(response.output)

[ResponseFunctionToolCall(arguments='{"total_leaves":24,"leaves_taken":8}', call_id='call_aGLCZ7I3KBE0bzRr0X9TqCTq', name='calculate_leave_balance', type='function_call', id='fc_089495b29be33e2d006aaccfd611cc87d095459a896450b674', async_=None, caller=None, namespace=None, status='completed')]


## **4.3 Execute the Requested Tool**

In [ ]:
import json

for tool_call in response.output:
    arguments = json.loads(tool_call.arguments)
    result = calculate_leave_balance(**arguments)

    print("==="*30)
    print("Tool:", tool_call.name)
    print("Arguments:", arguments)
    print("Result:", result)

Tool: calculate_leave_balance
Arguments: {'total_leaves': 24, 'leaves_taken': 8}
Result: {'remaining_leaves': 16}


# **5. Complete Tool Calling Workflow**


In [ ]:
input_items = []

for tool_call in response.output:

    if tool_call.type == "function_call":

        arguments = json.loads(tool_call.arguments)

        result = calculate_leave_balance(**arguments)

        input_items.append({
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result)
        })

final_response = client.responses.create(
    model=GENERATION_MODEL,
    previous_response_id=response.id,
    input=input_items,
    tools=tools
)

print(final_response.output_text)

The employee has a remaining leave balance of 16 days.


In [ ]:
input_items

[{'type': 'function_call_output',
  'call_id': 'call_aGLCZ7I3KBE0bzRr0X9TqCTq',
  'output': '{"remaining_leaves": 16}'}]

In [ ]:
input_items = [item.model_dump() for item in response.output]

for tool_call in response.output:
    arguments = json.loads(tool_call.arguments)
    result = calculate_leave_balance(**arguments)

    input_items.append({"type": "function_call_output",
                        "call_id": tool_call.call_id,
                        "output": json.dumps(result)
                        })
# print(input_items)

final_response = client.responses.create(model=GENERATION_MODEL,
                                         input=input_items,
                                         tools=tools)

final_response.output_text

### ======================================================